# Meta-Model PTRS Exploration — Tissue (39 GTEx) or Cell-Type (17 OneK1K)

Single parameterized notebook covering both:

- **`MODEL_VERSION='tissue'`** — 39 GTEx bulk-tissue PTRS, external = CAMP+1KG (drop 1KG via numeric IDs)
- **`MODEL_VERSION='ct'`** — 17 OneK1K cell-type PTRS, external = CAMP+GTEx (drop GTEx, keep `CA*` IDs)

Pipeline (identical for both versions):
1. Load PTRS for selected features from GACRS train+test
2. Stratified 75/25 train/test split (seed=0) + z-score normalization using train stats
3. Load PRS-CS for downstream comparison
4. Fit 10 tissue-combination methods for the unified PTRS (Simple Mean, Ridge x3, Lasso, Elastic Net, PCA x3, GB, RF GridSearch, Weighted Mean)
5. Load external cohort PTRS and apply train-derived z-scoring
6. Per-feature individual PTRS evaluated across 7 models on GACRS test + external full + CAMP-only balanced (64v64, 100 reps)
7. Re-evaluate the 7-model set on the unified-PTRS feature space (GACRS test + external full + balanced CAMP-only)
8. ROC curves

**Outputs saved to `data/predictions/meta_model_<MODEL_VERSION>/`:**
- `individual_ptrs_long.csv` — per-sample predictions for every (feature × model × cohort)
- `unified_ptrs_long.csv` — per-sample predictions for every (combination_method × cohort)
- `tissue_combination_results.csv` — Section 2 AUC table
- `unified_full_camp_results.csv` / `unified_balanced_results.csv` — Section 8 unified evaluations


## 0. Config — flip `MODEL_VERSION` and re-run

In [ ]:
# ============================================================
# INPUT_ROOT — where the input data lives (upstream PTRS + PRS profile).
# Expected layout under INPUT_ROOT:
#   INPUT_ROOT/results_concat_39_new/       (tissue focus summaries)
#   INPUT_ROOT/results_concat_17CT/         (CT focus summaries)
#   INPUT_ROOT/combine/data/1/ptrs_results-concat_39_*/   (tissue per-feature PTRS)
#   INPUT_ROOT/combine/data/1/ptrs_results-concat_17CT_*/ (CT per-feature PTRS)
#   INPUT_ROOT/combine/gacrs-only-with-1kg/*.profile      (PRS-CS profile)
#
# OUTPUT_ROOT — where this notebook writes results (this repo's data/predictions).
# Outputs go to OUTPUT_ROOT/meta_model_{tissue,ct}/
#
# This notebook covers the FOCUS-based unified PTRS pipeline only.
# For TWAS P+T mode (per-feature only, at various p-value thresholds),
# use `meta_model_exploration_TWAS_PT.ipynb`.
# ============================================================
from pathlib import Path
INPUT_ROOT         = Path('/Users/nancyh/Desktop/hartwell/gene_model/score')
OUTPUT_ROOT        = Path('/Users/nancyh/Desktop/asthma-prs-study-fresh/09_ptrs-unified_model-evaluation/data/predictions')

FOCUS_SUMMARY_ROOT = INPUT_ROOT
PTRS_ROOT          = INPUT_ROOT / 'combine'
PRS_ROOT           = INPUT_ROOT / 'combine'

# ---- pick which analysis to run ----
MODEL_VERSION = 'ct'      # 'tissue' or 'ct'  (set by runner)
# ------------------------------------

assert MODEL_VERSION in ('tissue', 'ct'), "MODEL_VERSION must be 'tissue' or 'ct'"

CONFIG = {
    'tissue': {
        'feature_label': 'tissue',
        'feature_label_plural': 'tissues',
        'feature_count_label': '39 GTEx tissues',
        'focus_summary_dir':   str(FOCUS_SUMMARY_ROOT / 'results_concat_39_new'),
        'focus_selected_file': 'selected_gacrs_val_all_case_gt_ctrl_keep.csv',
        'focus_ptrs_base':     str(PTRS_ROOT / 'data' / '1' / 'ptrs_results-concat_39_gacrs_train_test'),
        'camp_ptrs_base':      str(PTRS_ROOT / 'data' / '1' / 'ptrs_results-concat_39_camp_1k1k'),
        'external_label': 'CAMP+1KG',
        'removed_cohort_label': '1KG',
        'camp_only_mask_fn': lambda idx: ~idx.str.match(r'^\d'),  # drop numeric 1KG IDs
    },
    'ct': {
        'feature_label': 'cell_type',
        'feature_label_plural': 'cell_types',
        'feature_count_label': '17 OneK1K cell types',
        'focus_summary_dir':   str(FOCUS_SUMMARY_ROOT / 'results_concat_17CT'),
        'focus_selected_file': 'selected_17CT_gacrs_val_all_case_gt_ctrl_keep.csv',
        'focus_ptrs_base':     str(PTRS_ROOT / 'data' / '1' / 'ptrs_results-concat_17CT_gacrs_train_test'),
        'camp_ptrs_base':      str(PTRS_ROOT / 'data' / '1' / 'ptrs_results-concat_17CT_camp_gtex'),
        'external_label': 'CAMP+GTEx',
        'removed_cohort_label': 'GTEx',
        'camp_only_mask_fn': lambda idx: idx.str.startswith('CA'),  # keep CAMP IDs only
    },
}

cfg = dict(CONFIG[MODEL_VERSION])
cfg['focus_path'] = f"{cfg['focus_summary_dir']}/{cfg['focus_selected_file']}"

ARTIFACT_ROOT = OUTPUT_ROOT / f'meta_model_{MODEL_VERSION}'
for sub in ['predictions', 'bootstraps', 'models', 'feature_importance', 'config']:
    (ARTIFACT_ROOT / sub).mkdir(parents=True, exist_ok=True)
OUT_DIR = ARTIFACT_ROOT  # back-compat alias

print(f"INPUT_ROOT    = {INPUT_ROOT}")
print(f"OUTPUT_ROOT   = {OUTPUT_ROOT}")
print(f"MODEL_VERSION = {MODEL_VERSION}  ({cfg['feature_count_label']})")
print(f"Artifacts -> {ARTIFACT_ROOT}")


## 1. Imports & helpers

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
)
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.utils import resample
from sklearn.calibration import CalibratedClassifierCV
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns


def odds_ratio_quantile(y_true, y_pred, q=0.25):
    """Odds ratio: top quartile vs bottom quartile of predicted risk."""
    thresh_high = np.quantile(y_pred, 1 - q)
    thresh_low = np.quantile(y_pred, q)
    top = y_true[y_pred >= thresh_high]
    bottom = y_true[y_pred <= thresh_low]
    a = top.sum(); b = len(top) - a
    c = bottom.sum(); d = len(bottom) - c
    if b == 0 or c == 0:
        return np.inf
    return (a * d) / (b * c)


def append_predictions(records, sample_ids, scores, y_true, **labels):
    """Append per-sample prediction rows to a long-format list."""
    y_arr = np.asarray(y_true)
    s_arr = np.asarray(scores)
    for sid, s, y in zip(sample_ids, s_arr, y_arr):
        rec = {'sample_id': sid, 'score': float(s), 'y_true': int(y)}
        rec.update(labels)
        records.append(rec)


# === SAVE HELPERS (added by P+T patch) ===
import joblib, re

def safe_name(s):
    return re.sub(r'[^A-Za-z0-9_]+', '_', str(s)).strip('_')


def save_predictions(prefix, eval_set, sample_ids, y_true, preds):
    df = pd.DataFrame({
        'Sample_ID': list(sample_ids),
        'y_true':    np.asarray(y_true),
        'y_pred':    np.asarray(preds),
    })
    df.to_csv(ARTIFACT_ROOT / 'predictions' /
              f'{prefix}__{safe_name(eval_set)}.csv', index=False)


def save_bootstraps(prefix, aucs, ors, diffs=None):
    n = len(aucs)
    cols = {'iteration': list(range(n)), 'auc': aucs, 'or': ors}
    if diffs is not None:
        cols['mean_diff'] = diffs
    pd.DataFrame(cols).to_csv(
        ARTIFACT_ROOT / 'bootstraps' / f'{prefix}__camp_balanced_bootstraps.csv',
        index=False)


def save_model_and_importance(prefix, model, feature_names):
    joblib.dump(model, ARTIFACT_ROOT / 'models' / f'{prefix}.pkl')
    inner = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
    out = ARTIFACT_ROOT / 'feature_importance' / f'{prefix}.csv'
    if hasattr(inner, 'feature_importances_'):
        pd.DataFrame({'feature': list(feature_names),
                      'importance': inner.feature_importances_}).to_csv(out, index=False)
    elif hasattr(inner, 'coef_'):
        coef = inner.coef_[0] if inner.coef_.ndim > 1 else inner.coef_
        pd.DataFrame({'feature': list(feature_names),
                      'coefficient': np.asarray(coef)}).to_csv(out, index=False)


## 2. Load GACRS PTRS + train/test split + z-score normalize + load PRS-CS

In [ ]:
# Tissue/CT selection — always use the validation-set audit list from cfg['focus_path'].
# The consistency check (cell 21) then narrows this down to the cross-cohort
# consistent shortlist via thresholds on GACRS_AUC and CAMP_ONLY_BAL_AUC.
focus_summary = pd.read_csv(cfg['focus_path'])
col_name = 'Tissue_Type' if 'Tissue_Type' in focus_summary.columns else 'Tissue'
audit_features = sorted(focus_summary[col_name].dropna().unique())
focus_tissues = audit_features
focus_ptrs_base = cfg['focus_ptrs_base']
print(f"Validation-set audit ({cfg['focus_path']}): {len(focus_tissues)} {cfg['feature_label_plural']}")
for t in focus_tissues:
    print(f"  - {t}")

# Load per-feature PTRS for GACRS
ptrs_results = {}
for feat in focus_tissues:
    fp = Path(focus_ptrs_base) / f'{feat}_results.csv'
    if not fp.exists():
        print(f"Missing: {feat}")
        continue
    temp_df = pd.read_csv(fp, index_col=0)
    temp_df['Tissue'] = feat
    ptrs_results[feat] = temp_df

ptrs_df = pd.concat(ptrs_results.values())
ptrs_df = ptrs_df[['Tissue', 'Keep_Vector', 'asthma']]
ptrs_df.reset_index(inplace=True)

cases = ptrs_df[ptrs_df['asthma'] == 1]['Sample_ID'].drop_duplicates()
controls = ptrs_df[ptrs_df['asthma'] == 0]['Sample_ID'].drop_duplicates()

# Stratified split
all_samples = ptrs_df['Sample_ID'].drop_duplicates()
all_labels = ptrs_df.drop_duplicates('Sample_ID').set_index('Sample_ID')['asthma']
train_ids, test_ids = train_test_split(
    all_samples, test_size=0.25, stratify=all_labels.loc[all_samples], random_state=0
)

# Pivot to wide
ptrs = ptrs_df.set_index('Sample_ID')
balanced_ptrs = pd.concat([ptrs.loc[cases.values], ptrs.loc[controls.values]])
df_combineT = balanced_ptrs.pivot_table(index='Sample_ID', columns='Tissue', values='Keep_Vector')
asthma_info = balanced_ptrs[['asthma']].loc[~balanced_ptrs.index.duplicated(keep='first')]
df_combineT = df_combineT.merge(asthma_info, left_index=True, right_index=True)

tissue_cols = list(df_combineT.columns.difference(['asthma']))
combined_tissues = sorted(tissue_cols)
train_id = train_ids.values
test_id = test_ids.values

# Z-score normalize using train statistics
norm_stats = {}
for col in tissue_cols:
    m = df_combineT.loc[train_id, col].mean()
    s = df_combineT.loc[train_id, col].std()
    norm_stats[col] = {'mean': m, 'std': s}
    df_combineT[col] = (df_combineT[col] - m) / s

X_train = df_combineT.loc[train_id, combined_tissues]
X_test = df_combineT.loc[test_id, combined_tissues]
y_train = df_combineT.loc[train_id, 'asthma']
y_test = df_combineT.loc[test_id, 'asthma']

# Load PRS-CS (GACRS)
prscs_df = pd.read_csv(
    str(PRS_ROOT / 'gacrs-only-with-1kg' / 'gacrs_frz10_with-1kg_allchr_prscs_phi1e-04_mcmc85_1kg_eur_score.profile'),
    sep=r'\s+'
)[['IID', 'SCORESUM']].rename(columns={'IID': 'Sample_ID', 'SCORESUM': 'PRS_CS'}).set_index('Sample_ID')

df_with_prs = df_combineT.merge(prscs_df, left_index=True, right_index=True, how='inner')
valid_train = [s for s in train_id if s in df_with_prs.index]
valid_test = [s for s in test_id if s in df_with_prs.index]
prs_mean = df_with_prs.loc[valid_train, 'PRS_CS'].mean()
prs_std = df_with_prs.loc[valid_train, 'PRS_CS'].std()
df_with_prs['PRS_CS_z'] = (df_with_prs['PRS_CS'] - prs_mean) / prs_std
norm_stats_prscs = {'mean': prs_mean, 'std': prs_std}

print(f"\n{cfg['feature_label_plural'].capitalize()}: {len(combined_tissues)}")
print(f"Train: {len(train_id)} ({int(y_train.sum())} cases)")
print(f"Test:  {len(test_id)} ({int(y_test.sum())} cases)")
print(f"With PRS-CS: train={len(valid_train)}, test={len(valid_test)}")


## 3. Unified PTRS — 10 tissue-combination methods (fit on GACRS train, evaluate on GACRS test)

Collect OOF train predictions and test predictions for each method.


In [ ]:
# skf is shared by Section 6 per-feature RF GridSearch and Section 2 unified combinations.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


tissue_combination_results = []
# Stores method -> (train_oof, test_preds) for later saving
unified_method_preds = {}


def evaluate_unified(name, train_scores, test_scores):
    """Evaluate a unified PTRS score on the test set and record train+test preds."""
    auc = roc_auc_score(y_test, test_scores)
    grp1 = test_scores[y_test.values == 0]
    grp2 = test_scores[y_test.values == 1]
    _, p = ttest_ind(grp1, grp2, equal_var=False, nan_policy='omit')
    tissue_combination_results.append({
        'Method': name,
        'AUC': auc,
        'Mean_Diff': np.mean(grp2) - np.mean(grp1),
        'P_Value': p,
    })
    unified_method_preds[name] = {
        'train_oof': np.asarray(train_scores),
        'test_preds': np.asarray(test_scores),
    }
    return train_scores, test_scores


# ---- 1. Simple mean ----
evaluate_unified('Simple Mean', X_train.mean(axis=1).values, X_test.mean(axis=1).values)

# ---- 2-4. Ridge Logistic at three C values ----
for C in [0.01, 0.1, 1.0]:
    m = LogisticRegression(penalty='l2', C=C, solver='liblinear', max_iter=1000)
    m.fit(X_train, y_train)
    oof = cross_val_predict(
        LogisticRegression(penalty='l2', C=C, solver='liblinear', max_iter=1000),
        X_train, y_train, cv=skf, method='predict_proba'
    )[:, 1]
    evaluate_unified(f'Ridge Logistic (C={C})', oof, m.predict_proba(X_test)[:, 1])

# ---- 5. Lasso Logistic (L1, C=0.1) ----
lasso = LogisticRegression(penalty='l1', C=0.1, solver='liblinear', max_iter=1000)
lasso.fit(X_train, y_train)
lasso_oof = cross_val_predict(
    LogisticRegression(penalty='l1', C=0.1, solver='liblinear', max_iter=1000),
    X_train, y_train, cv=skf, method='predict_proba'
)[:, 1]
evaluate_unified('Lasso Logistic (C=0.1)', lasso_oof, lasso.predict_proba(X_test)[:, 1])
print(f"  Lasso non-zero {cfg['feature_label_plural']}: {(lasso.coef_[0] != 0).sum()}/{len(combined_tissues)}")

# ---- 6. Elastic Net (CV-tuned) ----
enet = LogisticRegressionCV(
    penalty='elasticnet', solver='saga', l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
    Cs=[0.001, 0.01, 0.1, 1.0], cv=5, max_iter=2000, random_state=42,
)
enet.fit(X_train, y_train)
enet_oof = cross_val_predict(
    LogisticRegressionCV(
        penalty='elasticnet', solver='saga', l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
        Cs=[0.001, 0.01, 0.1, 1.0], cv=5, max_iter=2000, random_state=42,
    ),
    X_train, y_train, cv=skf, method='predict_proba'
)[:, 1]
evaluate_unified('Elastic Net (CV)', enet_oof, enet.predict_proba(X_test)[:, 1])
print(f"  Best C={enet.C_[0]:.4f}, l1_ratio={enet.l1_ratio_[0]:.2f}")
print(f"  Non-zero {cfg['feature_label_plural']}: {(enet.coef_[0] != 0).sum()}/{len(combined_tissues)}")

# ---- 7. PCA -> Logistic at three n_components (filter out any n_comp > n_features) ----
pca_n_comps = sorted({n for n in [2, 5, min(10, len(combined_tissues))] if 0 < n <= len(combined_tissues)})
for n_comp in pca_n_comps:
    pipe = Pipeline([
        ('pca', PCA(n_components=n_comp)),
        ('lr', LogisticRegression(penalty='l2', C=0.1, solver='liblinear', max_iter=1000)),
    ])
    pipe.fit(X_train, y_train)
    oof = cross_val_predict(
        Pipeline([
            ('pca', PCA(n_components=n_comp)),
            ('lr', LogisticRegression(penalty='l2', C=0.1, solver='liblinear', max_iter=1000)),
        ]),
        X_train, y_train, cv=skf, method='predict_proba'
    )[:, 1]
    evaluate_unified(f'PCA({n_comp}) + Logistic', oof, pipe.predict_proba(X_test)[:, 1])

# ---- 8. Gradient Boosting ----
gb = GradientBoostingClassifier(
    n_estimators=100, max_depth=2, learning_rate=0.05,
    min_samples_leaf=20, subsample=0.8, random_state=42,
)
gb.fit(X_train, y_train)
gb_oof = cross_val_predict(
    GradientBoostingClassifier(
        n_estimators=100, max_depth=2, learning_rate=0.05,
        min_samples_leaf=20, subsample=0.8, random_state=42,
    ),
    X_train, y_train, cv=skf, method='predict_proba'
)[:, 1]
evaluate_unified('Gradient Boosting', gb_oof, gb.predict_proba(X_test)[:, 1])

# ---- 9. RF with GridSearch ----
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={
        'n_estimators': [50, 100, 200],
        'max_depth': [2, 3, 5, None],
        'min_samples_leaf': [10, 20, 50],
        'max_features': [1, 'sqrt', 'log2'],
    },
    cv=skf, scoring='roc_auc', n_jobs=-1, refit=True,
)
rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_
rf_oof = cross_val_predict(rf_best, X_train, y_train, cv=skf, method='predict_proba')[:, 1]
rf_test_pred = rf_best.predict_proba(X_test)[:, 1]
evaluate_unified('RF (GridSearch)', rf_oof, rf_test_pred)
print(f"  Best RF params: {rf_grid.best_params_}")

# ---- 10. Weighted mean by t-statistic ----
weights = {}
for col in combined_tissues:
    g1 = X_train.loc[y_train == 1, col]
    g0 = X_train.loc[y_train == 0, col]
    t_stat, _ = ttest_ind(g1, g0, equal_var=False)
    weights[col] = max(t_stat, 0)
total_w = sum(weights.values())
if total_w > 0:
    for k in weights:
        weights[k] /= total_w
    weighted_train = sum(X_train[col] * weights[col] for col in combined_tissues)
    weighted_test = sum(X_test[col] * weights[col] for col in combined_tissues)
    evaluate_unified('Weighted Mean (t-test)', weighted_train.values, weighted_test.values)
else:
    print('No positive feature signals — skipping weighted mean')

print('\nDone — all unified combination methods evaluated')

tissue_results_df = pd.DataFrame(tissue_combination_results).sort_values('AUC', ascending=False).reset_index(drop=True)
tissue_results_df.to_csv(ARTIFACT_ROOT / 'tissue_combination_results.csv', index=False)
print(f"Saved -> {ARTIFACT_ROOT / 'tissue_combination_results.csv'}")
tissue_results_df

In [ ]:
# Bar plot — unified combination methods on GACRS test
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ca02c' if r['AUC'] == tissue_results_df['AUC'].max() else '#1f77b4'
          for _, r in tissue_results_df.iterrows()]
ax.barh(range(len(tissue_results_df)), tissue_results_df['AUC'], color=colors)
ax.set_yticks(range(len(tissue_results_df)))
ax.set_yticklabels(tissue_results_df['Method'])
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
ax.set_xlabel('AUC')
ax.set_title(f'Unified PTRS Combination Methods — GACRS Test ({MODEL_VERSION})')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Define the 7 fixed models used for per-feature and unified evaluation

In [ ]:
# Shared model set used by Section 6 (per-feature) and Section 8 (unified PTRS)
tissue_models = {
    'Ridge (C=0.01)': LogisticRegression(penalty='l2', C=0.01, solver='liblinear', max_iter=1000),
    'Ridge (C=0.1)':  LogisticRegression(penalty='l2', C=0.1,  solver='liblinear', max_iter=1000),
    'Ridge (C=1.0)':  LogisticRegression(penalty='l2', C=1.0,  solver='liblinear', max_iter=1000),
    'Lasso (C=0.1)':  LogisticRegression(penalty='l1', C=0.1,  solver='liblinear', max_iter=1000),
    'Elastic Net':    LogisticRegressionCV(
        penalty='elasticnet', solver='saga', l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
        Cs=[0.001, 0.01, 0.1, 1.0], cv=5, max_iter=2000, random_state=42,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=2, learning_rate=0.05,
        min_samples_leaf=20, subsample=0.8, random_state=42,
    ),
    'RF (GridSearch)': GridSearchCV(
        RandomForestClassifier(random_state=42),
        param_grid={
            'n_estimators': [50, 100, 200],
            'max_depth': [2, 3, 5, None],
            'min_samples_leaf': [10, 20, 50],
            'max_features': [1, 'sqrt', 'log2'],
        },
        cv=skf, scoring='roc_auc', n_jobs=-1, refit=True,
    ),
}
print(f"Testing {len(tissue_models)} models for per-feature & unified evaluation")


## 5. Load external cohort PTRS and define CAMP-only subset

In [ ]:
# Load CAMP-only PTRS.
# Source file contains CAMP + non-CAMP samples (1KG for tissue / GTEx for CT);
# we load, apply the CAMP-only mask, and drop the non-CAMP subset immediately.
# Only X_camp_only / y_camp_only survive; no downstream cell evaluates on the
# full external cohort.
camp_ptrs_base = cfg['camp_ptrs_base']
camp_ptrs = {}
for feat in combined_tissues:
    fp = Path(camp_ptrs_base) / f'{feat}_results.csv'
    if not fp.exists():
        print(f"Missing: {feat}")
        continue
    df = pd.read_csv(fp, index_col=0)
    df['Tissue'] = feat
    camp_ptrs[feat] = df

camp_ptrs_df = pd.concat(camp_ptrs.values())
camp_ptrs_df = camp_ptrs_df[['Tissue', 'Keep_Vector', 'asthma']]
camp_ptrs_df.reset_index(inplace=True)

camp_combineT = camp_ptrs_df.pivot_table(index='Sample_ID', columns='Tissue', values='Keep_Vector')
camp_asthma = camp_ptrs_df.drop_duplicates('Sample_ID').set_index('Sample_ID')['asthma']
camp_combineT['asthma'] = camp_asthma
camp_combineT = camp_combineT.dropna(subset=['asthma'])

# Apply train-derived z-scoring
for col in combined_tissues:
    if col in norm_stats:
        camp_combineT[col] = (camp_combineT[col] - norm_stats[col]['mean']) / norm_stats[col]['std']

# Filter to CAMP-only immediately (drop 1KG / GTEx subcohort)
camp_only_mask = cfg['camp_only_mask_fn'](camp_combineT.index)
camp_combineT = camp_combineT[camp_only_mask]
X_camp_only = camp_combineT[combined_tissues]
y_camp_only = camp_combineT['asthma']
print(f"CAMP only (no {cfg['removed_cohort_label']}): {len(y_camp_only)} samples "
      f"({int(y_camp_only.sum())} cases, {int((y_camp_only==0).sum())} controls)")


## 6. Individual per-feature PTRS — violin plots (GACRS test / external full / CAMP-only)

In [ ]:
n_feat = len(combined_tissues)
fig, axes = plt.subplots(2, n_feat, figsize=(3 * max(n_feat, 1), 8), sharey=False)
if n_feat == 1:
    axes = axes.reshape(2, 1)

for j, ct in enumerate(combined_tissues):
    # Row 0: GACRS test
    g_scores = df_combineT.loc[test_id, ct]
    g_labels = df_combineT.loc[test_id, 'asthma']
    gp = pd.DataFrame({'PTRS': g_scores.values, 'Asthma': g_labels.values})
    gp['Asthma'] = gp['Asthma'].map({0: 'Control', 1: 'Case'})
    sns.violinplot(data=gp, x='Asthma', y='PTRS', ax=axes[0, j],
                   order=['Control', 'Case'], palette=['#4393c3', '#d6604d'], cut=0)
    _, p = ttest_ind(g_scores[g_labels == 0].values, g_scores[g_labels == 1].values, equal_var=False)
    axes[0, j].set_title(f'{ct}\nP={p:.3f}', fontsize=8)
    axes[0, j].set_xlabel('')
    axes[0, j].set_ylabel('PTRS (z)' if j == 0 else '')

    # Row 1: CAMP only
    if ct in X_camp_only.columns:
        co_scores = X_camp_only[ct]
        co_df = pd.DataFrame({'PTRS': co_scores.values, 'Asthma': y_camp_only.values})
        co_df['Asthma'] = co_df['Asthma'].map({0: 'Control', 1: 'Case'})
        sns.violinplot(data=co_df, x='Asthma', y='PTRS', ax=axes[1, j],
                       order=['Control', 'Case'], palette=['#4393c3', '#d6604d'], cut=0)
        _, cp = ttest_ind(co_scores[y_camp_only == 0].values, co_scores[y_camp_only == 1].values, equal_var=False)
        axes[1, j].set_title(f'P={cp:.3f}', fontsize=8)
    axes[1, j].set_xlabel('')
    axes[1, j].set_ylabel('PTRS (z)' if j == 0 else '')

row_labels = ['GACRS Test', 'CAMP Only']
for r, lbl in enumerate(row_labels):
    axes[r, 0].annotate(lbl, xy=(0, 0.5), xytext=(-0.8, 0.5),
                        xycoords='axes fraction', textcoords='axes fraction',
                        fontsize=12, fontweight='bold', rotation=90, va='center')
plt.suptitle(f"Individual {cfg['feature_label']} PTRS by case/control ({MODEL_VERSION})", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 7. Per-feature PTRS — 3-way evaluation across 7 models + save predictions

In [ ]:
# 2-way: GACRS test + balanced CAMP-only (downsampled cases, 100 reps)
individual_results = []
individual_predictions = []   # long-format per-sample scores (kept in memory)
individual_bootstraps  = {}   # {(feature, model): {'aucs': [...], 'ors': [...]}}
n_repeats = 100

camp_cases_all = X_camp_only[y_camp_only == 1]
camp_ctrls_all = X_camp_only[y_camp_only == 0]
n_ctrl = len(camp_ctrls_all)
print(f"Balanced CAMP-only: {n_ctrl} cases / {n_ctrl} controls × {n_repeats} bootstrap reps")

for ct in combined_tissues:
    X_tr = df_combineT.loc[train_id, [ct]]
    X_te = df_combineT.loc[test_id, [ct]]
    y_tr = df_combineT.loc[train_id, 'asthma']
    y_te = df_combineT.loc[test_id, 'asthma']

    camp_cases_ct = camp_cases_all[[ct]]
    camp_ctrls_ct = camp_ctrls_all[[ct]]

    for model_name, model_template in tissue_models.items():
        model = clone(model_template)
        model.fit(X_tr, y_tr)

        # GACRS test
        g_preds = model.predict_proba(X_te)[:, 1]
        g_auc = roc_auc_score(y_te, g_preds)
        g_or = odds_ratio_quantile(y_te.values, g_preds)
        _, g_p = ttest_ind(g_preds[y_te.values == 0], g_preds[y_te.values == 1], equal_var=False)
        append_predictions(individual_predictions, X_te.index.tolist(), g_preds, y_te.values,
                           feature=ct, model=model_name, cohort='GACRS_test', model_version=MODEL_VERSION)

        # CAMP only — single full pass (also save predictions)
        c_only_preds = model.predict_proba(X_camp_only[[ct]])[:, 1]
        _, co_p_feat = ttest_ind(c_only_preds[y_camp_only.values == 0], c_only_preds[y_camp_only.values == 1], equal_var=False, nan_policy='omit')
        append_predictions(individual_predictions, X_camp_only.index.tolist(), c_only_preds, y_camp_only.values,
                           feature=ct, model=model_name, cohort='CAMP_only', model_version=MODEL_VERSION)

        # CAMP only balanced bootstrap (for stable metric)
        c_aucs, c_ors = [], []
        for seed in range(n_repeats):
            cs = resample(camp_cases_ct, n_samples=n_ctrl, random_state=seed, replace=False)
            Xb = pd.concat([cs, camp_ctrls_ct])
            yb = pd.concat([y_camp_only[cs.index], y_camp_only[camp_ctrls_ct.index]])
            pb = model.predict_proba(Xb)[:, 1]
            c_aucs.append(roc_auc_score(yb, pb))
            c_ors.append(odds_ratio_quantile(yb.values, pb))
        c_bal_auc = float(np.mean(c_aucs))
        c_bal_std = float(np.std(c_aucs))
        c_bal_or = float(np.mean(c_ors))

        # Store bootstrap iterations IN MEMORY ONLY; the best-valid cell below
        # filters to (consistent feature × best classifier) and writes those to disk.
        individual_bootstraps[(ct, model_name)] = {'aucs': list(c_aucs), 'ors': list(c_ors)}

        individual_results.append({
            'Feature': ct, 'Model': model_name,
            'GACRS_AUC': g_auc, 'GACRS_OR': g_or, 'GACRS_P': g_p,
            'CAMP_ONLY_BAL_AUC': c_bal_auc,
            'CAMP_ONLY_BAL_AUC_std': c_bal_std,
            'CAMP_ONLY_BAL_OR': c_bal_or,
            'CAMP_ONLY_FULL_P': co_p_feat,
            'Delta_CAMP_ONLY_BAL_minus_GACRS': c_bal_auc - g_auc,
        })
    print(f"Done: {ct}")

individual_df = pd.DataFrame(individual_results)

# Per-sample predictions held in memory for the next cell.
individual_predictions_df = pd.DataFrame(individual_predictions)
print(f"  (per-sample predictions held in memory: {len(individual_predictions_df)} rows; "
        f'individual_bootstraps entries: {len(individual_bootstraps)})')


In [ ]:
individual_df

In [ ]:
# 2-way consistency table (filter: GACRS>0.5 AND CAMP-only bal>0.5)
consistency_df = individual_df.copy()
consistency_df['Consistency_Min_AUC']  = consistency_df[['GACRS_AUC', 'CAMP_ONLY_BAL_AUC']].min(axis=1)
consistency_df['Consistency_Mean_AUC'] = consistency_df[['GACRS_AUC', 'CAMP_ONLY_BAL_AUC']].mean(axis=1)
best_consistency = (
    consistency_df
    .sort_values(['Feature', 'Consistency_Min_AUC', 'Consistency_Mean_AUC'], ascending=[True, False, False])
    .drop_duplicates('Feature', keep='first')
)
best_consistency = best_consistency[
    (best_consistency['GACRS_AUC'] > 0.5) &
    (best_consistency['CAMP_ONLY_BAL_AUC'] > 0.5)
]
best_consistency = best_consistency.sort_values(['Consistency_Min_AUC', 'Consistency_Mean_AUC'], ascending=False).reset_index(drop=True)
print('=== 2-way consistency table (best model per feature, filtered) ===')
print(best_consistency[['Feature', 'Model', 'GACRS_AUC',
                        'CAMP_ONLY_BAL_AUC', 'CAMP_ONLY_BAL_AUC_std',
                        'Consistency_Min_AUC', 'Consistency_Mean_AUC']])


In [ ]:
individual_df.head()

In [ ]:
# === Best valid classifier per feature ===
# 1. Per (feature, model), compute Consistency_Min_AUC = min(GACRS, CAMP-only Bal)
#                              and Consistency_Mean_AUC = mean of the two.
# 2. For each feature, pick the classifier with the highest Consistency_Min_AUC
#    (tiebreak: Consistency_Mean_AUC) — legacy sort rule.
# 3. Filter: GACRS_AUC > 0.53 AND CAMP_ONLY_BAL_AUC > 0.52.
#    Reproduces the 6-tissue subset.
# 4. Sort final shortlist by Consistency_Min_AUC then Consistency_Mean_AUC, both descending.
#
# Saves:
#   {ARTIFACT_ROOT}/best_valid_per_feature.csv       full best-valid table (post-filter, sorted)
#   {ARTIFACT_ROOT}/consistent_features.csv          shortlist for downstream notebooks
#   {ARTIFACT_ROOT}/predictions/best_consistent_*.csv per-sample predictions
#   {ARTIFACT_ROOT}/bootstraps/best_consistent_*.csv  bootstrap iterations

# Step 1: add consistency columns
consistency_df = individual_df.copy()
consistency_df['Consistency_Min_AUC']  = consistency_df[['GACRS_AUC', 'CAMP_ONLY_BAL_AUC']].min(axis=1)
consistency_df['Consistency_Mean_AUC'] = consistency_df[['GACRS_AUC', 'CAMP_ONLY_BAL_AUC']].mean(axis=1)

# Step 2: sort + drop_duplicates → best classifier per feature by min-AUC consistency
best_valid_per_feature = (
    consistency_df
    .sort_values(['Feature', 'Consistency_Min_AUC', 'Consistency_Mean_AUC'],
                 ascending=[True, False, False])
    .drop_duplicates('Feature', keep='first')
)

# Step 3: 2-cohort threshold filter — `GACRS_AUC > 0.53 AND CAMP_ONLY_BAL_AUC > 0.52`.
GACRS_THRESH    = 0.53
CAMP_BAL_THRESH = 0.52
best_valid_per_feature = best_valid_per_feature[
    (best_valid_per_feature['GACRS_AUC']        > GACRS_THRESH) &
    (best_valid_per_feature['CAMP_ONLY_BAL_AUC'] > CAMP_BAL_THRESH)
]

# Step 4: final sort
best_valid_per_feature = best_valid_per_feature.sort_values(
    ['Consistency_Min_AUC', 'Consistency_Mean_AUC'], ascending=False
).reset_index(drop=True)

cols_show = ['Feature', 'Model', 'GACRS_AUC',
             'CAMP_ONLY_BAL_AUC', 'CAMP_ONLY_BAL_AUC_std',
             'Consistency_Min_AUC', 'Consistency_Mean_AUC']
print(f"=== Best classifier per {cfg['feature_label']} — {len(best_valid_per_feature)} {cfg['feature_label_plural']} ===")
print(best_valid_per_feature[cols_show].to_string(index=False))

all_feats = set(individual_df['Feature'].unique())
kept = set(best_valid_per_feature['Feature'].unique())
dropped = sorted(all_feats - kept)
if dropped:
    print(f"\n  Features dropped by filter: {dropped}")
else:
    print(f"\n  All {len(kept)} {cfg['feature_label_plural']} passed the filter")

# 1) Full best-valid table + consistent shortlist
best_valid_per_feature.to_csv(ARTIFACT_ROOT / 'best_valid_per_feature.csv', index=False)
consistent_features = best_valid_per_feature[['Feature', 'Model',
    'GACRS_AUC', 'CAMP_ONLY_BAL_AUC', 'CAMP_ONLY_BAL_AUC_std',
    'Consistency_Min_AUC', 'Consistency_Mean_AUC']].copy()
consistent_features.to_csv(ARTIFACT_ROOT / 'consistent_features.csv', index=False)
print(f"Saved -> {ARTIFACT_ROOT / 'best_valid_per_feature.csv'}")
print(f"Saved -> {ARTIFACT_ROOT / 'consistent_features.csv'}  "
      f"({len(consistent_features)} consistent {cfg['feature_label_plural']})")

# 2) Per-sample predictions (per cohort: GACRS_test, CAMP_only) & per-iteration
#    bootstraps for (best classifier × consistent feature).
predictions_long_df = pd.DataFrame(individual_predictions)
n_pred_files, n_boot_files = 0, 0
for _, row in best_valid_per_feature.iterrows():
    feat, mdl = row['Feature'], row['Model']
    prefix = f'best_consistent__{safe_name(feat)}__{safe_name(mdl)}'
    sub = predictions_long_df[(predictions_long_df['feature'] == feat) &
                              (predictions_long_df['model']   == mdl)]
    for cohort in sub['cohort'].unique():
        chunk = sub[sub['cohort'] == cohort]
        save_predictions(prefix, cohort,
                         chunk['sample_id'].tolist(),
                         chunk['y_true'].values,
                         chunk['score'].values)
        n_pred_files += 1
    bs = individual_bootstraps.get((feat, mdl))
    if bs is not None:
        out_b = ARTIFACT_ROOT / 'bootstraps' / f'{prefix}__camp_balanced.csv'
        pd.DataFrame({'iteration': range(len(bs['aucs'])),
                      'auc': bs['aucs'], 'or': bs['ors']}).to_csv(out_b, index=False)
        n_boot_files += 1

print(f"Saved {n_pred_files} per-cohort per-sample prediction CSV(s) and "
      f"{n_boot_files} bootstrap CSV(s) under {ARTIFACT_ROOT}/predictions and /bootstraps")

In [ ]:
# === Subset feature matrices to the consistent shortlist before Section 9 unified eval ===
# Sections 1-7 evaluated every audit feature so the best-valid cell could pick a
# cross-cohort consistent shortlist. From here on (Sections 9-onwards) the unified
# PTRS should be built on that shortlist only, so the predictions match what
# integrated_ptrs_prs_unified.ipynb will compute downstream.
consistent_feature_list = sorted(consistent_features['Feature'].tolist())
print(f"Restricting Section 9 unified PTRS to {len(consistent_feature_list)} consistent "
      f"{cfg['feature_label_plural']}:")
for t in consistent_feature_list:
    print(f"  - {t}")

X_train     = X_train[consistent_feature_list]
X_test      = X_test[consistent_feature_list]
X_camp_only = X_camp_only[consistent_feature_list]
combined_tissues = consistent_feature_list
print(f"X_train shape now: {X_train.shape}")

## 8. Cohort confound sanity check (CAMP vs the removed subcohort)

In [ ]:
# Cell removed: external cohort composition audit is no longer relevant
# (X_camp / y_camp were dropped after CAMP-only filtering in cell 13).


## 9. Unified PTRS — 7-model evaluation on GACRS test + external full + CAMP-only balanced + save predictions

In [ ]:
# Balanced CAMP-only for the 7 unified-PTRS models
balanced_results = []
unified_predictions = []  # long-format per-sample scores for each unified method

# 3a) Per-model unified predictions on GACRS test + CAMP-only balanced
for tissue_name, tissue_model in tissue_models.items():
    tm = clone(tissue_model)
    tm.fit(X_train, y_train)
    save_model_and_importance(f'unified_section9__{safe_name(tissue_name)}',
                              tm, combined_tissues)

    # GACRS test
    g_preds = tm.predict_proba(X_test)[:, 1]
    g_auc = roc_auc_score(y_test, g_preds)
    g_or = odds_ratio_quantile(y_test.values, g_preds)
    g1 = g_preds[y_test.values == 0]; g2 = g_preds[y_test.values == 1]
    _, g_p = ttest_ind(g1, g2, equal_var=False, nan_policy='omit')
    append_predictions(unified_predictions, X_test.index.tolist(), g_preds, y_test.values,
                       method=tissue_name, cohort='GACRS_test', model_version=MODEL_VERSION)

    # CAMP only — single pass (record predictions) and balanced bootstrap (for stable metric)
    co_preds = tm.predict_proba(X_camp_only)[:, 1]
    c1 = co_preds[y_camp_only.values == 0]; c2 = co_preds[y_camp_only.values == 1]
    _, co_p = ttest_ind(c1, c2, equal_var=False, nan_policy='omit')
    append_predictions(unified_predictions, X_camp_only.index.tolist(), co_preds, y_camp_only.values,
                       method=tissue_name, cohort='CAMP_only', model_version=MODEL_VERSION)

    # Balanced bootstrap
    camp_cases = X_camp_only[y_camp_only == 1]
    camp_controls = X_camp_only[y_camp_only == 0]
    n_controls = len(camp_controls)
    aucs, ors, diffs = [], [], []
    for seed in range(n_repeats):
        cs = resample(camp_cases, n_samples=n_controls, random_state=seed, replace=False)
        Xb = pd.concat([cs, camp_controls])
        yb = pd.concat([y_camp_only[cs.index], y_camp_only[camp_controls.index]])
        pb = tm.predict_proba(Xb)[:, 1]
        aucs.append(roc_auc_score(yb, pb))
        ors.append(odds_ratio_quantile(yb.values, pb))
        b1 = pb[yb.values == 0]; b2 = pb[yb.values == 1]
        diffs.append(b2.mean() - b1.mean())
    balanced_results.append({
        'Method': tissue_name,
        'Eval_Set': f'CAMP-only Balanced ({n_controls}v{n_controls})',
        'AUC': float(np.mean(aucs)), 'AUC_std': float(np.std(aucs)),
        'OR': float(np.mean(ors)), 'OR_std': float(np.std(ors)),
        'Mean_Diff': float(np.mean(diffs)),
        'GACRS_AUC': g_auc, 'GACRS_OR': g_or, 'GACRS_P': g_p,
        'CAMP_ONLY_FULL_P': co_p,
    })
    save_bootstraps(f'unified_section9__{safe_name(tissue_name)}', aucs, ors, diffs)

# 3b) Also save unified-method predictions from Section 3 (the 10 combination methods on GACRS only)
for method_name, preds in unified_method_preds.items():
    append_predictions(unified_predictions,
                       X_train.index.tolist(), preds['train_oof'], y_train.values,
                       method=f'{method_name} (Section-3)', cohort='GACRS_train_OOF',
                       model_version=MODEL_VERSION)
    append_predictions(unified_predictions,
                       X_test.index.tolist(), preds['test_preds'], y_test.values,
                       method=f'{method_name} (Section-3)', cohort='GACRS_test',
                       model_version=MODEL_VERSION)

unified_predictions_df = pd.DataFrame(unified_predictions)
unified_predictions_df.to_csv(ARTIFACT_ROOT / 'predictions' / 'unified_ptrs_long.csv', index=False)
print(f"Saved -> {ARTIFACT_ROOT / 'predictions' / 'unified_ptrs_long.csv'}  ({len(unified_predictions_df)} rows)")

balanced_df = pd.DataFrame(balanced_results).sort_values('AUC', ascending=False)
balanced_df.to_csv(ARTIFACT_ROOT / 'unified_balanced_results.csv', index=False)
print(f"Saved -> {ARTIFACT_ROOT / 'unified_balanced_results.csv'}")

print(f"\n=== Balanced CAMP-only ({n_controls} per class, {n_repeats} reps) ===")
print(balanced_df[['Method', 'AUC', 'AUC_std', 'OR', 'OR_std', 'Mean_Diff']].to_string(index=False))

## 10. ROC curves — unified PTRS only

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
for tissue_name, tissue_model in tissue_models.items():
    tm = clone(tissue_model)
    tm.fit(X_train, y_train)
    g_preds = tm.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, g_preds)
    ax.plot(fpr, tpr, label=f"{tissue_name} ({roc_auc_score(y_test, g_preds):.3f})")
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'Unified PTRS Only — GACRS Test ({MODEL_VERSION})')
ax.legend(fontsize=7, loc='lower right')
plt.tight_layout()
plt.show()


## 11. Summary

- `MODEL_VERSION='tissue'`: 39-GTEx-tissue PTRS — moderate within-cohort discrimination
- `MODEL_VERSION='ct'`: 17-OneK1K-cell-type PTRS — weaker within-cohort signal
- Per-feature reporting includes 3-way table: GACRS test + external full + CAMP-only balanced
- RF GridSearch consistently captures the best within-cohort signal across both versions
- External CAMP-only balanced evaluation is the most robust transfer estimate

Predictions and metric tables for this run are saved under `data/predictions/meta_model_<MODEL_VERSION>/`. Flip `MODEL_VERSION` in cell 0 and re-run to produce the other variant.


In [ ]:
# === CONFIG DUMP ===
# Writes ARTIFACT_ROOT/config/config.json summarizing the inputs & params used
# for this run, so each per-pval artifact directory is self-describing.
def _ser(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating,)): return float(v)
    if isinstance(v, (np.ndarray,)): return v.tolist()
    return v

config_summary = {
    'MODEL_VERSION': MODEL_VERSION,
    'feature_count_label': cfg['feature_count_label'],
    'features_used': combined_tissues,
    'cohort_sizes': {
        'gacrs_train':  int(len(train_id)),
        'gacrs_test':   int(len(test_id)),
        'camp_only':    int(len(X_camp_only)),
    },
    'random_seeds': {
        'train_test_split_random_state':   0,
        'cv_StratifiedKFold_random_state': 42,
        'bootstrap_seed_range':            [0, n_repeats],
    },
    'norm_stats_features': {
        k: {'mean': float(v['mean']), 'std': float(v['std'])}
        for k, v in norm_stats.items()
    },
    'norm_stats_prscs': {'mean': float(norm_stats_prscs['mean']),
                         'std':  float(norm_stats_prscs['std'])},
    'unified_methods_section3': list(unified_method_preds.keys()),
    'unified_models_section9':  list(tissue_models.keys()),
    'paths': {
        'focus_path':      cfg['focus_path'],
        'focus_ptrs_base': cfg['focus_ptrs_base'],
        'camp_ptrs_base':  cfg['camp_ptrs_base'],
    },
}

import json as _json
with open(ARTIFACT_ROOT / 'config' / 'config.json', 'w') as f:
    _json.dump(config_summary, f, indent=2, default=str)

print(f"Config dump -> {ARTIFACT_ROOT / 'config' / 'config.json'}")
print(f"\nArtifact summary:")
for sub in ['predictions', 'bootstraps', 'models', 'feature_importance', 'config']:
    n = len(list((ARTIFACT_ROOT / sub).glob('*')))
    print(f"  {sub:22s} {n:4d} file(s)")